In [ ]:
# WITHIN-SUBJECT
import mne
import torch
from moabb.datasets import BNCI2014_001

dataset = BNCI2014_001()
data = dataset.get_data()

subjects = list(data.keys())


def epochs_from_session(session_dict, tmin=-0.2, tmax=2.0):

    raws = list(session_dict.values())
    raw = mne.concatenate_raws(raws) if len(raws) > 1 else raws[0]

    events, event_id = mne.events_from_annotations(raw)
    epochs = mne.Epochs(raw, events, event_id=event_id,
                         tmin=tmin, tmax=tmax, baseline=None, preload=True)
    epochs.pick('eeg')
    return epochs


# dicionário: subject_id -> {"train": epochs_sessao0, "test": epochs_sessao1}
within_subject_tensors = {}

for subj in subjects:
    session_keys = list(data[subj].keys())

    train_epochs = epochs_from_session(data[subj][session_keys[0]])
    test_epochs = epochs_from_session(data[subj][session_keys[1]])

    X_tr = train_epochs.get_data() * 1e6
    y_tr = train_epochs.events[:, -1]
    X_te = test_epochs.get_data() * 1e6
    y_te = test_epochs.events[:, -1]

    min_lbl = y_tr.min()

    within_subject_tensors[subj] = {
        "X_train": torch.tensor(X_tr, dtype=torch.float32).unsqueeze(1),
        "y_train": torch.tensor(y_tr - min_lbl, dtype=torch.long),
        "X_test": torch.tensor(X_te, dtype=torch.float32).unsqueeze(1),
        "y_test": torch.tensor(y_te - min_lbl, dtype=torch.long),
    }

for subj, tensors in within_subject_tensors.items():
    X_train_sub = tensors["X_train"]
    y_train_sub = tensors["y_train"]
    X_test_sub = tensors["X_test"]
    y_test_sub = tensors["y_test"]

Used Annotations descriptions: [np.str_('feet'), np.str_('left_hand'), np.str_('right_hand'), np.str_('tongue')]
Not setting metadata
288 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 288 events and 551 original time points ...
0 bad epochs dropped
Used Annotations descriptions: [np.str_('feet'), np.str_('left_hand'), np.str_('right_hand'), np.str_('tongue')]
Not setting metadata
288 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 288 events and 551 original time points ...
0 bad epochs dropped
Used Annotations descriptions: [np.str_('feet'), np.str_('left_hand'), np.str_('right_hand'), np.str_('tongue')]
Not setting metadata
288 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 288 events and 551 original time points ...
0 bad epochs dropped
Used Annotations descriptions: [np.str_('fe